In [1]:


from src.agents.nodes.sql_generate_query_node import generate_sql_node
from utils.sql_tools import WmsSqlTool
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool, QuerySQLCheckerTool, InfoSQLDatabaseTool,ListSQLDatabaseTool
from langchain_google_genai import ChatGoogleGenerativeAI
from src.config import settings

_llm = None

def _getllm() -> ChatGoogleGenerativeAI:
    global _llm

    if _llm is None:
        _llm = ChatGoogleGenerativeAI(
            model=settings.GOOGLE_AI_MODEL,
            api_key=settings.GOOGLE_API_KEY,


        )
    return _llm
tools = WmsSqlTool(query_check_llm=_getllm())

query_tool = next(t for t in tools.get_sql_tools() if isinstance(t, QuerySQLDatabaseTool))
query_check = next(t for t in tools.get_sql_tools() if isinstance(t, QuerySQLCheckerTool))
list_tables = next(t for t in tools.get_sql_tools() if isinstance(t, ListSQLDatabaseTool))
list_info = next(t for t in tools.get_sql_tools() if isinstance(t, InfoSQLDatabaseTool))

res= list_info.invoke({"table_names": list_tables.invoke("")})


ImportError: cannot import name 'generate_sql_node' from 'src.agents.nodes.sql_generate_query_node' (/Users/rahul/Library/Mobile Documents/com~apple~CloudDocs/SCM_Agentic_Ai/WMS_Incident_Copilot/src/agents/nodes/sql_generate_query_node.py)

In [ ]:
res

In [ ]:
tool = tools.get_sql_tools()

next(t for t in tool if isinstance(t, QuerySQLDatabaseTool))

In [ ]:
from pprint import pprint
check_sql = "select * from inventory wher limit 10"
checked_sql = query_check.invoke({"query": check_sql})
pprint(checked_sql)


In [ ]:
from data.state import WMState
from agents.nodes.router import router_node

state = WMState(
    ticket_number= "INC12345",
    description= "how to slot sku003 and how much",
    user_id= "rahul"
)

router = router_node(state)

router


In [ ]:
from agents.graph.sql_subgraph import sql_graph
from domain.states.sql_subgraph_state.sql_graph_state import SQLGraphState
from dotenv import load_dotenv
load_dotenv()
state = SQLGraphState(

    description= "how many picks left?",
    domain="outbound"
)

router = sql_graph.invoke(state)

from rich.console import Console
from rich.markdown import Markdown
console = Console()

console.print(Markdown(router['final_response']))


In [1]:
from langgraph.graph import StateGraph, START, END
from src.data.state import WMState
from IPython.display import display, Image
from src.agents.nodes.router import router_node
from agents.graph.sql_subgraph import sql_graph
from src.agents.nodes.sql_result_node import return_result_node
from src.agents.edges.router_intent_edge import router_intent_edge
from langgraph.cache.memory import InMemoryCache
from agents.nodes.supervisor_node import SupervisorNode
from agents.nodes.inbound_agent_node import inbound_agent_node
from agents.nodes.outbound_agent_node import outbound_agent_node
from agents.nodes.inventory_agent_node import inventory_agent_node
from dotenv import load_dotenv
load_dotenv()
builder = StateGraph(WMState)
supervisor_node = (SupervisorNode())

builder.add_node("router_node", router_node)
builder.add_node("sql_query_subgraph", sql_graph)
builder.add_node("return_result_node", return_result_node)
builder.add_node("supervisor_node", supervisor_node)
builder.add_node("inbound_agent_node", inbound_agent_node)
builder.add_node("outbound_agent_node", outbound_agent_node)
builder.add_node("inventory_agent_node", inventory_agent_node)


builder.add_edge(START, "router_node")
builder.add_conditional_edges(
    "router_node",
    router_intent_edge,
    {
        "lookup":"sql_query_subgraph",
        "diagnose": "supervisor_node"
    }
)
builder.add_conditional_edges(
    "supervisor_node",
    lambda state: ["inbound_agent_node", "outbound_agent_node", "inventory_agent_node"],
    {
        "inbound_agent_node": "inbound_agent_node",
        "outbound_agent_node": "outbound_agent_node",
        "inventory_agent_node": "inventory_agent_node",
    }
)

# Agents → back to supervisor
builder.add_edge("inbound_agent_node", "supervisor_node")
builder.add_edge("outbound_agent_node", "supervisor_node")
builder.add_edge("inventory_agent_node", "supervisor_node")

# Supervisor can also go to result
builder.add_edge("return_result_node", END)

graph = builder.compile(cache=InMemoryCache())

display(Image(graph.get_graph().draw_mermaid_png()))


ModuleNotFoundError: No module named 'src.data.state'

In [ ]:
result = await graph.ainvoke(
    {
        "ticket_number": "INC12345",
        "description": "what all issues in inbound ?",
        "user_id": "rahul",
        "domain": "inventory",
    }
)

from rich.console import Console
from rich.markdown import Markdown

console = Console()

print(type(result))
print(result)

final_text = result.get("final_response") or str(result)
print("FINAL_TEXT:", repr(final_text))

console.print(Markdown(final_text))

In [ ]:


from langchain_core.messages import HumanMessage

from data.state import WMState
from models.model_loader import get_google_llm

get_google_llm().invoke([HumanMessage(content="Hello")]).content[0]["text"].strip()

In [ ]:


from agents.nodes.supervisor_node import SupervisorNode
from data.state import WMState

message = (
   "picking is low and uph is low but invenotry is high why and also inbound has no work"
)

state = WMState(
    ticket_number="INC12345",
    description=message,
    user_id="rahul",
)


node = SupervisorNode()
response = await node(state)
print(response)


In [ ]:
for data in response.goto:
    print(data)
    print("="*90)


In [ ]:
from agents.nodes.supervisor_node import SupervisorNode
from agents.nodes.inbound_agent_node import inbound_agent_node
from data.state import WMState, WorkerInput

AGENTS = {
    "inbound_agent": inbound_agent_node,
    # "outbound_agent": outbound_agent_node,
    # "inventory_agent": inventory_agent_node,
}

message = (
    "whats the over wms load"
)

state = WMState(
    ticket_number="INC12345",
    description=message,
    user_id="rahul",
)

supervisor = SupervisorNode()

# Run supervisor
cmd = await supervisor(state)
print("Supervisor returned:")
print(cmd)


In [ ]:
from agents.graph.sql_subgraph import sql_graph
from data.state import SQLGraphState
state2 = SQLGraphState(
    domain="outbound",
    description="how much left in picking outbound and what all skus"
)

res = sql_graph.invoke(state2)

res

In [ ]:
state2 = SQLGraphState(
    domain="outbound",
    description="how much sku do we need for picking"
)

res = sql_graph.invoke(state2)

res

In [ ]:
from models.model_loader import get_openai_fast_llm
from domain.states.supervisor.supervisor_subagent_task_state import SupervisorToSubAgentDeligationItem

llm = get_openai_fast_llm().with_structured_output(SupervisorToSubAgentDeligationItem)

response = llm.invoke([
    {"role": "system", "content": "You are a WMS diagnose agent elaborate issues in depth"},
    {"role": "user", "content": "whats my inbound issues"}
])

response

In [ ]:
for data in response.subagent_deligations:
    print(data.subagent_task)
    print("="*19)

In [ ]:
from agents.nodes.supervisor_node import SupervisorNode
from domain.states.supervisor.diagnose_graph_state import WMState

message = (
    "whats inbound staus"
)
state = WMState(
    ticket_number="INC12345",
    description=message,
)


node = SupervisorNode()
response = await node(state)
print(response)


# subagent_name= subagent_name,
#             worker_task=subagent_research_task,
#             task_id=str(task_id),
#             domain_name=domain,

In [ ]:
response.goto

In [ ]:
from langgraph.graph import StateGraph, START, END
from domain.states.supervisor.diagnose_graph_state import WMState
from IPython.display import display, Image
from src.agents.nodes.router import router_node
from agents.graph.sql_subgraph import sql_graph
from agents.edges.router_intent_edge import router_intent_edge
from agents.nodes.supervisor_node import WarehouseSupervisorNode
from agents.nodes.diagnose_result_node import diagnose_result_node
from agents.nodes.sql_lookup_subgraph_node import sql_query_subgraph_node
from dotenv import load_dotenv
load_dotenv()
builder = StateGraph(WMState)
supervisor_node = WarehouseSupervisorNode()

builder.add_node("router_node", router_node)
builder.add_node("sql_query_subgraph_node", sql_query_subgraph_node)

builder.add_node("supervisor_node", supervisor_node)
builder.add_node("diagnose_result_node", diagnose_result_node)

builder.add_edge(START, "router_node")

builder.add_conditional_edges(
    "router_node",
    router_intent_edge,
    {
        "lookup": "sql_query_subgraph_node",
        "diagnose": "supervisor_node",
    },
)

# lookup path ends here
builder.add_edge("sql_query_subgraph_node", END)

# diagnose_result_node is your explicit finish node
builder.add_edge("diagnose_result_node", END)

graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:

result = await graph.ainvoke(
    {
        "ticket_number":"INC12345",
        "description":"what status of inbound ?",
    }
)

from rich.console import Console
from rich.markdown import Markdown

console = Console()

print(type(result))
print(result)

final_text = result.get("final_responses") or str(result)

console.print(Markdown(final_text['final_responses']))

In [ ]:
final_text['final_responses']

In [ ]:
from agents.nodes.router import router_node
from domain.states.supervisor.diagnose_graph_state import WMState

state = WMState(
    ticket_number="INC12345",
    description="what is the sttus of sku 003",
    session_id="rahul"
)

res = router_node(state)

print(res)

In [ ]:
from langgraph.graph import StateGraph
from domain.states.sql_subgraph_state.sql_graph_state import SQLGraphState
from IPython.display import display, Image
from domain.states.sql_subgraph_state.sql_graph_state import SQLGraphState
from src.agents.nodes.sql_load_skills_node import sql_load_skills_node
from src.agents.nodes.sql_generate_query_node import sql_generate_query_node
from src.agents.nodes.check_sql_node import check_sql_node
from src.agents.nodes.sql_run_sql_node import sql_run_sql_node
from src.agents.nodes.sql_result_node import sql_result_node
from langgraph.graph import StateGraph, START, END

sql_graph_state = StateGraph(SQLGraphState)
builder = sql_graph_state.add_sequence([
        sql_load_skills_node,
        sql_generate_query_node,
        check_sql_node,
        sql_run_sql_node,
        sql_result_node,
    ])

sql_graph_state.add_edge(START, "sql_load_skills_node")
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
example = SQLGraphState(
    domain="outbound",
    user_question="whats the total picks left?",
)

result = graph.invoke(example)

import pandas as pd

df = pd.DataFrame(result.get('result').rows)
df

In [ ]:
result.get('result').rows

In [13]:
from runtime.session_runtime import WMSSessionRuntime
from agents.graph.application_graph import graph
runtime = WMSSessionRuntime(graph)

# First run
result = await runtime.run(
    session_id="prajwal",
    ticket_number="INC123454343",
    description="how much sku 004 i have?"
)

# Second run — same session, history saved
# result = await runtime.run(
#     session_id="rahul",
#     user_input="now check outbound picking"
# )
print(result)

Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('domain.states.sql_generate_subquery.sql_generate_subqueries_state', 'GenerateSubqueries')]


Saved: /Users/rahul/Library/Mobile Documents/com~apple~CloudDocs/SCM_Agentic_Ai/WMS_Incident_Copilot/src/data/sessions/prajwal.json
{'ticket_number': 'INC123454343', 'session_id': 'prajwal', 'description': 'how much sku 004 i have?', 'final': True, 'routing_decision': {'intent': 'lookup', 'domain': ['inventory']}, 'lookup_result': {'domain': ['inventory'], 'parent_session_id': 'prajwal', 'user_question': 'how much sku 004 i have?', 'skill_context': [{'inventory': "\n# Inventory Lookup\n# Follow guideline strictly\n## Tables\n\n### wms1.inventory\n- id (PRIMARY KEY)\n- case_number (UNIQUE)\n- sku\n- unit_qty\n- location\n- wrkref\n- dtl_num\n- ins_dt\n\n## Use Cases\n- How much inventory do we have for a SKU?\n- Where is a SKU stored?\n- Which cases contain a SKU?\n- What inventory is tied to a work reference or detail number?\n\n## Business Rules\n- unit_qty is the quantity stored in a case-level inventory record.\n- Total on-hand inventory for a SKU = SUM(unit_qty) from wms1.inventory

In [22]:
resultt = await runtime.run(
    session_id="prajwal",
    ticket_number="INC123454343",
    description="how much sku 004 i have?"
)
result


Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('domain.states.sql_generate_subquery.sql_generate_subqueries_state', 'GenerateSubqueries')]


Saved: /Users/rahul/Library/Mobile Documents/com~apple~CloudDocs/SCM_Agentic_Ai/WMS_Incident_Copilot/src/data/sessions/prajwal.json


{'ticket_number': 'INC123454343',
 'session_id': 'prajwal',
 'description': 'how much sku 004 i have?',
 'final': True,
 'routing_decision': {'intent': 'lookup', 'domain': ['inventory']},
 'lookup_result': {'domain': ['inventory'],
  'parent_session_id': 'prajwal',
  'user_question': 'how much sku 004 i have?',
  'skill_context': [{'inventory': "\n# Inventory Lookup\n# Follow guideline strictly\n## Tables\n\n### wms1.inventory\n- id (PRIMARY KEY)\n- case_number (UNIQUE)\n- sku\n- unit_qty\n- location\n- wrkref\n- dtl_num\n- ins_dt\n\n## Use Cases\n- How much inventory do we have for a SKU?\n- Where is a SKU stored?\n- Which cases contain a SKU?\n- What inventory is tied to a work reference or detail number?\n\n## Business Rules\n- unit_qty is the quantity stored in a case-level inventory record.\n- Total on-hand inventory for a SKU = SUM(unit_qty) from wms1.inventory.\n- Inventory can be grouped by sku, location, case_number, wrkref, or dtl_num.\n- case_number is unique per inventory r

In [3]:
from agents.graph.application_graph import graph

graph = graph

graph.invoke({
     "ticket_number":"INC12345",
    "session_id":"prajwal",
    "user_id":"prajwal",
    "description":"how much sku 004 i have? ?",
}, config={"configurable": {"thread_id":"prajwal"}})

2026-04-09 23:14:45,240 | INFO | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-09 23:14:46,515 | INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
2026-04-09 23:14:46,540 | INFO | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-09 23:14:47,677 | INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
2026-04-09 23:14:47,682 | INFO | src.agents.nodes.sql_generate_query_node | SQL Generated Query Node
2026-04-09 23:14:52,071 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 23:14:56,028 | INFO | src.agents.nodes.sql_run_sql_node | SQL NODE: Sql ran on the database


{'ticket_number': 'INC12345',
 'session_id': 'prajwal',
 'user_id': 'prajwal',
 'description': 'how much sku 004 i have? ?',
 'status': 'done',
 'event_log': [{'node': 'supervisor_node', 'status': 'Completed'},
  {'node': 'sql_query_subgraph_node', 'status': 'done'}],
 'errors': [],
 'current_node': 'sql_query_subgraph_node',
 'final': True,
 'routing_decision': {'intent': 'lookup', 'domain': ['inventory']},
 'lookup_result': {'domain': ['inventory'],
  'parent_session_id': 'prajwal',
  'user_question': 'how much sku 004 i have? ?',
  'skill_context': [{'inventory': "\n# Inventory Lookup\n# Follow guideline strictly\n## Tables\n\n### wms1.inventory\n- id (PRIMARY KEY)\n- case_number (UNIQUE)\n- sku\n- unit_qty\n- location\n- wrkref\n- dtl_num\n- ins_dt\n\n## Use Cases\n- How much inventory do we have for a SKU?\n- Where is a SKU stored?\n- Which cases contain a SKU?\n- What inventory is tied to a work reference or detail number?\n\n## Business Rules\n- unit_qty is the quantity stored i

In [4]:
config={"configurable": {"thread_id":"prajwal"}}
history = list(graph.get_state_history(config))

for i, s in enumerate(history):
    print(f"\n--- checkpoint {i} ---")
    print("config:", s.config)
    print("next:", s.next)
    print("values:", s.values)  # includes checkpoint info

2026-04-09 23:15:08,872 | WARNING | langgraph.checkpoint.serde.jsonplus | Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('domain.states.sql_generate_subquery.sql_generate_subqueries_state', 'GenerateSubqueries')]
2026-04-09 23:15:08,873 | WARNING | langgraph.checkpoint.serde.jsonplus | Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('domain.states.sql_generate_subquery.sql_generate_subqueries_state', 'GenerateSubqueries')]
2026-04-09 23:15:08,873 | WARNING | langgraph.checkpoint.serde.jsonplus | Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be block


--- checkpoint 0 ---
config: {'configurable': {'thread_id': 'prajwal', 'checkpoint_ns': '', 'checkpoint_id': '1f134a49-8586-6cfe-8008-8159fe93158d'}}
next: ()
values: {'ticket_number': 'INC12345', 'session_id': 'prajwal', 'user_id': 'prajwal', 'description': 'how much sku 004 i have? ?', 'status': 'done', 'event_log': [{'node': 'supervisor_node', 'status': 'Completed'}, {'node': 'sql_query_subgraph_node', 'status': 'done'}], 'errors': [], 'current_node': 'sql_query_subgraph_node', 'final': True, 'routing_decision': {'intent': 'lookup', 'domain': ['inventory']}, 'lookup_result': {'domain': ['inventory'], 'parent_session_id': 'prajwal', 'user_question': 'how much sku 004 i have? ?', 'skill_context': [{'inventory': "\n# Inventory Lookup\n# Follow guideline strictly\n## Tables\n\n### wms1.inventory\n- id (PRIMARY KEY)\n- case_number (UNIQUE)\n- sku\n- unit_qty\n- location\n- wrkref\n- dtl_num\n- ins_dt\n\n## Use Cases\n- How much inventory do we have for a SKU?\n- Where is a SKU stored?\n

In [5]:
history = list(graph.get_state_history(config))
old_snapshot = history[8]   # example: choose one checkpoint

replay_result = graph.invoke(None, old_snapshot.config)
print(replay_result)

2026-04-09 23:15:30,558 | WARNING | langgraph.checkpoint.serde.jsonplus | Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('domain.states.sql_generate_subquery.sql_generate_subqueries_state', 'GenerateSubqueries')]
2026-04-09 23:15:30,559 | WARNING | langgraph.checkpoint.serde.jsonplus | Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('domain.states.sql_generate_subquery.sql_generate_subqueries_state', 'GenerateSubqueries')]
2026-04-09 23:15:30,559 | WARNING | langgraph.checkpoint.serde.jsonplus | Deserializing unregistered type domain.states.sql_generate_subquery.sql_generate_subqueries_state.GenerateSubqueries from checkpoint. This will be block

The putaway process for inbound orders involves moving products from the receiving dock to their designated storage locations. According to standard warehouse operating procedures, the process generally follows these steps:

### 1. Preparation and Setup
*   **Documentation:** Ensure that incoming items have appropriate lot labels or barcode labels applied. If the warehouse uses an Advanced Shipping Notification (ASN), these labels may be available before the load arrives. If not, they are generated upon the driver's arrival.
*   **Staging:** While some operations move product directly from the dock to racks, others may use a staging area. If a load is not fully stowed, forklift operators may combine quantities on the dock with those already in storage to expedite the carrier's departure.

### 2. Putaway Execution
*   **Assignment:** Depending on the level of automation, pallet positions are either pre-assigned using "directed putaway" (often managed by a WMS) or manually determined by 

2026-04-09 23:15:42,595 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The putaway process for inbound orders involves several key steps:

1. **Preparation and Setup**: Ensure all documentation is in order, including lot or barcode labels. Use a staging area if necessary.

2. **Putaway Execution**: Assign storage locations, either through a WMS or manually, and transport products to their designated areas.

3. **Consistent Putaway Logic**: Follow a logical order for placement, grouping lots, product codes, and customer products together.

4. **Special Considerations**: Address temperature sensitivity for perishable items and optimize placement for fast-moving products.

5. **Finalizing the Process**: Verify the stowed product and update inventory records, creating a receiving manifest for any discrepancies.

If you have any specific questions or need further details, please let me know!

2026-04-09 23:15:45,621 | INFO | agents.nodes.supervisor_node | SUPERVISOR NODE CALLED


{'ticket_number': 'INC12345', 'session_id': 'prajwal', 'user_id': 'prajwal', 'description': 'how to putaway inbound orders ?', 'status': 'done', 'event_log': [{'node': 'supervisor_node', 'status': 'Completed'}], 'errors': [], 'final': True, 'routing_decision': {'intent': 'diagnose', 'domain': ['inbound']}, 'diagnosis_result': 'The putaway process for inbound orders involves moving products from the receiving dock to their designated storage locations. According to standard warehouse operating procedures, the process generally follows these steps:\n\n### 1. Preparation and Setup\n*   **Documentation:** Ensure that incoming items have appropriate lot labels or barcode labels applied. If the warehouse uses an Advanced Shipping Notification (ASN), these labels may be available before the load arrives. If not, they are generated upon the driver\'s arrival.\n*   **Staging:** While some operations move product directly from the dock to racks, others may use a staging area. If a load is not fu